# ShopStream — Exploración y ETL en EMR Studio

Notebook para validar interactivamente el pipeline antes de promover el código a `etl_main.py` (spark-submit).

**Requisitos**:
- Cluster EMR conectado (Spark 3.5.0).
- Kernel: PySpark.
- Acceso S3 a los buckets de ShopStream.

## 1. Setup — descargar el código del proyecto desde S3

In [ ]:
import os
import subprocess

# Descarga el zip del paquete spark_jobs y lo añade al PYTHONPATH
CODE_BUCKET = 'shopstream-processed-mf0106'
subprocess.check_call(['aws', 's3', 'cp', f's3://{CODE_BUCKET}/code/spark_jobs.zip', '/tmp/spark_jobs.zip'])
import sys
sys.path.insert(0, '/tmp/spark_jobs.zip')
print('PYTHONPATH:', sys.path[:3])

## 2. SparkSession y carga de datos

In [ ]:
from spark_jobs.session import build_spark_session, read_events

spark = build_spark_session('shopstream-exploratory')
spark.sparkContext.setLogLevel('WARN')

RAW_BUCKET = 's3://shopstream-raw-mf0106/events'
PROCESSED_BUCKET = 's3://shopstream-processed-mf0106/metrics'

# Lee un día específico para iterar rápido
raw = read_events(spark, f'{RAW_BUCKET}/year=2026/month=05/day=22/*.jsonl')
raw.printSchema()
raw.show(5, truncate=False)
print('Total raw:', raw.count())

## 3. Análisis de calidad antes de limpiar

In [ ]:
from pyspark.sql import functions as F

# Conteo por tipo de evento
raw.groupBy('event_type').count().orderBy(F.desc('count')).show()

# Nulos por columna (sólo en page_view, donde country/device_type aplican)
pv = raw.filter(F.col('event_type') == 'page_view')
pv.select(
    F.sum(F.col('country').isNull().cast('int')).alias('null_country'),
    F.sum(F.col('device_type').isNull().cast('int')).alias('null_device'),
    F.sum(F.col('time_on_page_seconds').isNull().cast('int')).alias('null_time'),
    F.sum(F.col('event_id').isNull().cast('int')).alias('null_event_id'),
).show()

# event_id duplicados (deberían ser cero en un día puro)
duplicates = raw.groupBy('event_id').count().filter(F.col('count') > 1).count()
print(f'Duplicate event_ids: {duplicates}')

## 4. Limpieza

In [ ]:
from spark_jobs.cleaning import clean_events

clean = clean_events(raw).cache()
print('Clean events:', clean.count())
clean.show(3, truncate=False)

## 5. Métricas individuales

In [ ]:
from spark_jobs.metrics import (
    top_pages, bounce_rate, funnel, product_gap, nav_paths, device_country, anomalies
)

print('=== Top pages ===')
top_pages.compute(clean).show(10, truncate=False)

print('=== Bounce rate ===')
bounce_rate.compute(clean).show(truncate=False)

print('=== Funnel ===')
funnel.compute(clean).show(truncate=False)

print('=== Product gap (top 10) ===')
product_gap.compute(clean).show(10, truncate=False)

print('=== Nav paths ===')
nav_paths.compute(clean).show(truncate=False)

print('=== Device × Country ===')
device_country.compute(clean).show(20, truncate=False)

print('=== Anomalies ===')
anomalies.compute(clean).show(10, truncate=False)

## 6. Ejecutar el ETL completo (escribe Parquet)

Equivalente a `spark-submit etl_main.py`. Útil para validar el output completo desde el notebook.

In [ ]:
from spark_jobs.etl_main import run

summary = run(
    input_base=RAW_BUCKET,
    output_base=PROCESSED_BUCKET,
    run_date='2026-05-22',
)
summary

## 7. Verificar outputs en S3

In [ ]:
for metric in ['top_pages', 'bounce_rate', 'funnel', 'product_gap',
               'nav_paths', 'device_country', 'anomalies']:
    df = spark.read.parquet(f'{PROCESSED_BUCKET}/{metric}/dt=2026-05-22')
    print(f'\n=== {metric} ({df.count()} rows) ===')
    df.show(5, truncate=False)